# Login Data Quality Cleaner - Phase 1 Milestone Project

A batch Python script that reads a messy raw login export, flags rows with data quality problems, writes a clean CSV, and generates a data quality report (terminal and .txt).

What this notebook does:
- Validates IP addresses, timestamps, and status values against custom rules
- Detects duplicate logins: same user and IP within a 1-minute window
- Normalizes fields that pass validation but arrive in inconsistent raw form (case, whitespace, mixed date formats), writing the clean form back before the row is saved
- Excludes flagged rows entirely from the clean output
- Reports issue counts to terminal and .txt, counting each bad row once even if it trips multiple issues
- Documents the build step by step: the initial version, each bug found once it ran against real data, the fix applied, and the reasoning behind it

Built with: Python, csv module, datetime, rich (Console, Table, Panel)


In [ ]:
import csv
import os
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

INPUT_FILE = "portfolio/milestone_project/messy_logins.csv"
OUTPUT_FILE = "portfolio/milestone_project/clean_logins.csv"
REPORT_FILE = "portfolio/milestone_project/data_quality_report.txt"

REQUIRED_FIELDS = ["timestamp", "user_id", "ip_address", "status"]
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",     # 2026-06-08 08:12:34
    "%d/%m/%Y %H:%M:%S",     # 08/06/2026 08:14:02
    "%Y-%m-%dT%H:%M:%SZ",    # 2026-06-08T08:18:00Z
]
VALID_STATUSES = {"SUCCESS", "FAILED"}

console = Console(record=True)


def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def is_valid_ip(ip):
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not 0 <= int(part) <= 255:
            return False
    return True


def parse_timestamp(ts_string):
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(ts_string, fmt)
        except ValueError:
            continue
    return None


def is_valid_status(status):
    return status.strip().upper() in VALID_STATUSES


clean_rows = []
seen = set()

total_rows = 0
flagged_rows = 0

issue_counts = {
    "missing_fields": 0,
    "invalid_ip": 0,
    "invalid_timestamp": 0,
    "invalid_status": 0,
    "duplicate": 0,
}

with open(INPUT_FILE, "r", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames

    clear_screen()
    console.print(Panel.fit(
        "        MILESTONE PROJECT - LOGIN DATA CLEANING        ",
        style="bold white on blue",
        border_style="blue"
    ))
    console.print()

    for row in reader:
        total_rows += 1
        issues = set()

        user_value = (row.get("user_id") or "").strip().lower()
        ip_value = (row.get("ip_address") or "").strip()
        ts_value = (row.get("timestamp") or "").strip()
        status_value = (row.get("status") or "").strip()

        if not user_value or not ip_value or not ts_value or not status_value:
            issues.add("missing_fields")

        if ip_value and not is_valid_ip(ip_value):
            issues.add("invalid_ip")

        parsed_ts = None
        if ts_value:
            parsed_ts = parse_timestamp(ts_value)
            if parsed_ts is None:
                issues.add("invalid_timestamp")

        if status_value and not is_valid_status(status_value):
            issues.add("invalid_status")

        if user_value and ip_value and parsed_ts:
            rounded_ts = parsed_ts.replace(second=0, microsend=0)
            dup_key = (user_value, ip_value, rounded_ts)

            if dup_key in seen:
                issues.add("duplicate")
            else:
                seen.add(dup_key)

        if issues:
            flagged_rows += 1
            for issue in issues:
                issue_counts[issue] += 1
        else:
            clean_rows.append(row)

    console.print(f"TOTAL ROWS READ: {total_rows}")
    console.print(f"{len(clean_rows)} clean vs {flagged_rows} flagged\n")


with open(OUTPUT_FILE, "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

console.print(f"Clean data written to {OUTPUT_FILE}")

console.print("DATA QUALITY REPORT")
console.print("-" * 50)

report_table = Table(box=None)
report_table.add_column("Issue Type", style="bold yellow")
report_table.add_column("Rows Affected", justify="right", style="bold red")

for issue_label, count in issue_counts.items():
    report_table.add_row(issue_label.replace("_", " ").title(), str(count))

console.print(report_table)

flagged_pct = (flagged_rows / total_rows * 100) if total_rows else 0
clean_pct = (len(clean_rows) / total_rows * 100) if total_rows else 0

console.print(Panel(
    f"Total Rows Processed: {total_rows}\n"
    f"Clean Rows Written:   {len(clean_rows)} ({clean_pct:.1f}%)\n"
    f"Flagged Rows:         {flagged_rows} ({flagged_pct:.1f}%)\n"
    f"(a flagged row is counted once here, even if it tripped multiple issue types above)",
    title="Summary",
    border_style="cyan"
))

console.save_text(REPORT_FILE)


### Revision 1: Bug Check - misspelled keyword argument
* Ran the script against my real `messy_logins.csv` for the first time and got: `TypeError: 'microsend' is an invalid keyword argument for replace()`.
* Traced it to the duplicate-detection line: I'd misspelled `microsecond` as `microsend` when typing the rounding logic.
* Consulted Claude Chat to confirm: `datetime.replace()` only accepts its exact named fields (`year`, `month`, `day`, `hour`, `minute`, `second`, `microsecond`), so a misspelled keyword raises a `TypeError` immediately, it doesn't get treated as a typo of an existing field.

**The fix:**
* Corrected `microsend=0` to `microsecond=0`.
* This line is what turns a full-precision timestamp into a rounded "to the minute" one. Without it working, the 1-minute duplicate window can't be calculated at all and the script can't run past that point.

Updated script:


In [ ]:
import csv
import os
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

INPUT_FILE = "portfolio/milestone_project/messy_logins.csv"
OUTPUT_FILE = "portfolio/milestone_project/clean_logins.csv"
REPORT_FILE = "portfolio/milestone_project/data_quality_report.txt"

REQUIRED_FIELDS = ["timestamp", "user_id", "ip_address", "status"]
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",     # 2026-06-08 08:12:34
    "%d/%m/%Y %H:%M:%S",     # 08/06/2026 08:14:02
    "%Y-%m-%dT%H:%M:%SZ",    # 2026-06-08T08:18:00Z
]
VALID_STATUSES = {"SUCCESS", "FAILED"}

console = Console(record=True)


def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def is_valid_ip(ip):
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not 0 <= int(part) <= 255:
            return False
    return True


def parse_timestamp(ts_string):
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(ts_string, fmt)
        except ValueError:
            continue
    return None


def is_valid_status(status):
    return status.strip().upper() in VALID_STATUSES


clean_rows = []
seen = set()

total_rows = 0
flagged_rows = 0

issue_counts = {
    "missing_fields": 0,
    "invalid_ip": 0,
    "invalid_timestamp": 0,
    "invalid_status": 0,
    "duplicate": 0,
}

with open(INPUT_FILE, "r", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames

    clear_screen()
    console.print(Panel.fit(
        "        MILESTONE PROJECT - LOGIN DATA CLEANING        ",
        style="bold white on blue",
        border_style="blue"
    ))
    console.print()

    for row in reader:
        total_rows += 1
        issues = set()

        user_value = (row.get("user_id") or "").strip().lower()
        ip_value = (row.get("ip_address") or "").strip()
        ts_value = (row.get("timestamp") or "").strip()
        status_value = (row.get("status") or "").strip()

        if not user_value or not ip_value or not ts_value or not status_value:
            issues.add("missing_fields")

        if ip_value and not is_valid_ip(ip_value):
            issues.add("invalid_ip")

        parsed_ts = None
        if ts_value:
            parsed_ts = parse_timestamp(ts_value)
            if parsed_ts is None:
                issues.add("invalid_timestamp")

        if status_value and not is_valid_status(status_value):
            issues.add("invalid_status")

        if user_value and ip_value and parsed_ts:
            rounded_ts = parsed_ts.replace(second=0, microsecond=0)
            dup_key = (user_value, ip_value, rounded_ts)

            if dup_key in seen:
                issues.add("duplicate")
            else:
                seen.add(dup_key)

        if issues:
            flagged_rows += 1
            for issue in issues:
                issue_counts[issue] += 1
        else:
            clean_rows.append(row)

    console.print(f"TOTAL ROWS READ: {total_rows}")
    console.print(f"{len(clean_rows)} clean vs {flagged_rows} flagged\n")


with open(OUTPUT_FILE, "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

console.print(f"Clean data written to {OUTPUT_FILE}")

console.print("DATA QUALITY REPORT")
console.print("-" * 50)

report_table = Table(box=None)
report_table.add_column("Issue Type", style="bold yellow")
report_table.add_column("Rows Affected", justify="right", style="bold red")

for issue_label, count in issue_counts.items():
    report_table.add_row(issue_label.replace("_", " ").title(), str(count))

console.print(report_table)

flagged_pct = (flagged_rows / total_rows * 100) if total_rows else 0
clean_pct = (len(clean_rows) / total_rows * 100) if total_rows else 0

console.print(Panel(
    f"Total Rows Processed: {total_rows}\n"
    f"Clean Rows Written:   {len(clean_rows)} ({clean_pct:.1f}%)\n"
    f"Flagged Rows:         {flagged_rows} ({flagged_pct:.1f}%)\n"
    f"(a flagged row is counted once here, even if it tripped multiple issue types above)",
    title="Summary",
    border_style="cyan"
))

console.save_text(REPORT_FILE)


### Revision 2: Bug Check - normalized value not written back to output
* Script ran clean this time. Output: 5 clean rows, 3 flagged (missing_fields: 2, invalid_status: 1, everything else: 0).
* Looked closely at `clean_logins.csv` and noticed `"  admin  "` (from a row with extra whitespace in the original CSV) made it into the clean output with the whitespace still there, even though `user_value` was normalized (`.strip().lower()`) for the validation logic.
* Root cause: `user_value` was only ever used to build the duplicate-check key and the missing-fields check. It was never written back into `row["user_id"]`, so the original raw dict value is what actually gets saved by the CSV writer.

**The fix:**
* Added `row["user_id"] = user_value` immediately after building `user_value`.
* Since `row` is a dict and `clean_rows.append(row)` stores a reference to that same dict, mutating it early means whatever's in `row["user_id"]` at append-time is what reaches the output file. No changes needed in the writer step itself.

Updated script:


In [ ]:
import csv
import os
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

INPUT_FILE = "portfolio/milestone_project/messy_logins.csv"
OUTPUT_FILE = "portfolio/milestone_project/clean_logins.csv"
REPORT_FILE = "portfolio/milestone_project/data_quality_report.txt"

REQUIRED_FIELDS = ["timestamp", "user_id", "ip_address", "status"]
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",     # 2026-06-08 08:12:34
    "%d/%m/%Y %H:%M:%S",     # 08/06/2026 08:14:02
    "%Y-%m-%dT%H:%M:%SZ",    # 2026-06-08T08:18:00Z
]
VALID_STATUSES = {"SUCCESS", "FAILED"}

console = Console(record=True)


def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def is_valid_ip(ip):
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not 0 <= int(part) <= 255:
            return False
    return True


def parse_timestamp(ts_string):
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(ts_string, fmt)
        except ValueError:
            continue
    return None


def is_valid_status(status):
    return status.strip().upper() in VALID_STATUSES


clean_rows = []
seen = set()

total_rows = 0
flagged_rows = 0

issue_counts = {
    "missing_fields": 0,
    "invalid_ip": 0,
    "invalid_timestamp": 0,
    "invalid_status": 0,
    "duplicate": 0,
}

with open(INPUT_FILE, "r", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames

    clear_screen()
    console.print(Panel.fit(
        "        MILESTONE PROJECT - LOGIN DATA CLEANING        ",
        style="bold white on blue",
        border_style="blue"
    ))
    console.print()

    for row in reader:
        total_rows += 1
        issues = set()

        user_value = (row.get("user_id") or "").strip().lower()
        row["user_id"] = user_value  # write normalized value back so clean output reflects it
        ip_value = (row.get("ip_address") or "").strip()
        ts_value = (row.get("timestamp") or "").strip()
        status_value = (row.get("status") or "").strip()

        if not user_value or not ip_value or not ts_value or not status_value:
            issues.add("missing_fields")

        if ip_value and not is_valid_ip(ip_value):
            issues.add("invalid_ip")

        parsed_ts = None
        if ts_value:
            parsed_ts = parse_timestamp(ts_value)
            if parsed_ts is None:
                issues.add("invalid_timestamp")

        if status_value and not is_valid_status(status_value):
            issues.add("invalid_status")

        if user_value and ip_value and parsed_ts:
            rounded_ts = parsed_ts.replace(second=0, microsecond=0)
            dup_key = (user_value, ip_value, rounded_ts)

            if dup_key in seen:
                issues.add("duplicate")
            else:
                seen.add(dup_key)

        if issues:
            flagged_rows += 1
            for issue in issues:
                issue_counts[issue] += 1
        else:
            clean_rows.append(row)

    console.print(f"TOTAL ROWS READ: {total_rows}")
    console.print(f"{len(clean_rows)} clean vs {flagged_rows} flagged\n")


with open(OUTPUT_FILE, "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

console.print(f"Clean data written to {OUTPUT_FILE}")

console.print("DATA QUALITY REPORT")
console.print("-" * 50)

report_table = Table(box=None)
report_table.add_column("Issue Type", style="bold yellow")
report_table.add_column("Rows Affected", justify="right", style="bold red")

for issue_label, count in issue_counts.items():
    report_table.add_row(issue_label.replace("_", " ").title(), str(count))

console.print(report_table)

flagged_pct = (flagged_rows / total_rows * 100) if total_rows else 0
clean_pct = (len(clean_rows) / total_rows * 100) if total_rows else 0

console.print(Panel(
    f"Total Rows Processed: {total_rows}\n"
    f"Clean Rows Written:   {len(clean_rows)} ({clean_pct:.1f}%)\n"
    f"Flagged Rows:         {flagged_rows} ({flagged_pct:.1f}%)\n"
    f"(a flagged row is counted once here, even if it tripped multiple issue types above)",
    title="Summary",
    border_style="cyan"
))

console.save_text(REPORT_FILE)


### Revision 3: Bug Check - same normalization gap in status
* Same question applied to `status`: the last row had `"FAILED "` (trailing space) which passes `is_valid_status()` since it strips before checking, but still lands in the clean CSV with the trailing space intact.
* This isn't just cosmetic. Project 5 (`meridian_login_analysis.py`) does `if row["status"] == "FAILED":`, a strict string comparison. `"FAILED "` fails that check silently and falls into the "unknown status" branch instead of being counted, no error, no warning, just a silently wrong count if this clean file ever fed a script written in that style.

**The fix:**
* Changed `status_value` to normalize with `.strip().upper()` instead of just `.strip()`.
* Added `row["status"] = status_value` right after, same write-back pattern as `user_id`.

Updated script:


In [ ]:
import csv
import os
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

INPUT_FILE = "portfolio/milestone_project/messy_logins.csv"
OUTPUT_FILE = "portfolio/milestone_project/clean_logins.csv"
REPORT_FILE = "portfolio/milestone_project/data_quality_report.txt"

REQUIRED_FIELDS = ["timestamp", "user_id", "ip_address", "status"]
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",     # 2026-06-08 08:12:34
    "%d/%m/%Y %H:%M:%S",     # 08/06/2026 08:14:02
    "%Y-%m-%dT%H:%M:%SZ",    # 2026-06-08T08:18:00Z
]
VALID_STATUSES = {"SUCCESS", "FAILED"}

console = Console(record=True)


def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def is_valid_ip(ip):
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not 0 <= int(part) <= 255:
            return False
    return True


def parse_timestamp(ts_string):
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(ts_string, fmt)
        except ValueError:
            continue
    return None


def is_valid_status(status):
    return status.strip().upper() in VALID_STATUSES


clean_rows = []
seen = set()

total_rows = 0
flagged_rows = 0

issue_counts = {
    "missing_fields": 0,
    "invalid_ip": 0,
    "invalid_timestamp": 0,
    "invalid_status": 0,
    "duplicate": 0,
}

with open(INPUT_FILE, "r", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames

    clear_screen()
    console.print(Panel.fit(
        "        MILESTONE PROJECT - LOGIN DATA CLEANING        ",
        style="bold white on blue",
        border_style="blue"
    ))
    console.print()

    for row in reader:
        total_rows += 1
        issues = set()

        user_value = (row.get("user_id") or "").strip().lower()
        row["user_id"] = user_value  # write normalized value back so clean output reflects it
        ip_value = (row.get("ip_address") or "").strip()
        ts_value = (row.get("timestamp") or "").strip()
        status_value = (row.get("status") or "").strip().upper()
        row["status"] = status_value  # write normalized value back so clean output reflects it

        if not user_value or not ip_value or not ts_value or not status_value:
            issues.add("missing_fields")

        if ip_value and not is_valid_ip(ip_value):
            issues.add("invalid_ip")

        parsed_ts = None
        if ts_value:
            parsed_ts = parse_timestamp(ts_value)
            if parsed_ts is None:
                issues.add("invalid_timestamp")

        if status_value and not is_valid_status(status_value):
            issues.add("invalid_status")

        if user_value and ip_value and parsed_ts:
            rounded_ts = parsed_ts.replace(second=0, microsecond=0)
            dup_key = (user_value, ip_value, rounded_ts)

            if dup_key in seen:
                issues.add("duplicate")
            else:
                seen.add(dup_key)

        if issues:
            flagged_rows += 1
            for issue in issues:
                issue_counts[issue] += 1
        else:
            clean_rows.append(row)

    console.print(f"TOTAL ROWS READ: {total_rows}")
    console.print(f"{len(clean_rows)} clean vs {flagged_rows} flagged\n")


with open(OUTPUT_FILE, "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

console.print(f"Clean data written to {OUTPUT_FILE}")

console.print("DATA QUALITY REPORT")
console.print("-" * 50)

report_table = Table(box=None)
report_table.add_column("Issue Type", style="bold yellow")
report_table.add_column("Rows Affected", justify="right", style="bold red")

for issue_label, count in issue_counts.items():
    report_table.add_row(issue_label.replace("_", " ").title(), str(count))

console.print(report_table)

flagged_pct = (flagged_rows / total_rows * 100) if total_rows else 0
clean_pct = (len(clean_rows) / total_rows * 100) if total_rows else 0

console.print(Panel(
    f"Total Rows Processed: {total_rows}\n"
    f"Clean Rows Written:   {len(clean_rows)} ({clean_pct:.1f}%)\n"
    f"Flagged Rows:         {flagged_rows} ({flagged_pct:.1f}%)\n"
    f"(a flagged row is counted once here, even if it tripped multiple issue types above)",
    title="Summary",
    border_style="cyan"
))

console.save_text(REPORT_FILE)


### Revision 4: Bug Check - whitespace and mixed date formats surviving into output
* Asked myself the same question for the two remaining fields:
  1. `ip_address`: stripped for validation (`ip_value = ... .strip()`) but never written back, so a whitespace-padded IP would slip through the same way `user_id` did. No example in my data, but the risk is identical.
  2. `timestamp`: the real issue. My clean CSV had three different timestamp string formats sitting in the same column (`2026-06-08 08:12:34`, `08/06/2026 08:14:02`, `2026-06-08T08:18:00Z`) because the script accepts all three as valid but never standardizes them. This is worse than the case and whitespace issues, it's not cosmetic, it's structurally broken for any future sorting, filtering, or `pd.to_datetime()` call on that column.

**The fix:**
* Added `row["ip_address"] = ip_value` right after building `ip_value`, same pattern as before.
* For `timestamp`, since `parsed_ts` is already a real `datetime` object once parsing succeeds, wrote it back out in one canonical format: `row["timestamp"] = parsed_ts.strftime("%Y-%m-%d %H:%M:%S")`.

**Design takeaway:**
* The rule that emerged isn't "normalize everything," it's: if a field passes validation but its raw form could still reach the output in an inconsistent state, write the clean form back before it's saved. `action` doesn't need this since it isn't validated at all, and duplicate rows don't need it since they're excluded from the output entirely.

Final script:


In [ ]:
import csv
import os
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

INPUT_FILE = "portfolio/milestone_project/messy_logins.csv"
OUTPUT_FILE = "portfolio/milestone_project/clean_logins.csv"
REPORT_FILE = "portfolio/milestone_project/data_quality_report.txt"

REQUIRED_FIELDS = ["timestamp", "user_id", "ip_address", "status"]
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",     # 2026-06-08 08:12:34
    "%d/%m/%Y %H:%M:%S",     # 08/06/2026 08:14:02
    "%Y-%m-%dT%H:%M:%SZ",    # 2026-06-08T08:18:00Z
]
VALID_STATUSES = {"SUCCESS", "FAILED"}

console = Console(record=True)


def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')


def is_valid_ip(ip):
    parts = ip.split(".")
    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        if not 0 <= int(part) <= 255:
            return False
    return True


def parse_timestamp(ts_string):
    for fmt in TIMESTAMP_FORMATS:
        try:
            return datetime.strptime(ts_string, fmt)
        except ValueError:
            continue
    return None


def is_valid_status(status):
    return status.strip().upper() in VALID_STATUSES


clean_rows = []
seen = set()

total_rows = 0
flagged_rows = 0

issue_counts = {
    "missing_fields": 0,
    "invalid_ip": 0,
    "invalid_timestamp": 0,
    "invalid_status": 0,
    "duplicate": 0,
}

with open(INPUT_FILE, "r", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames

    clear_screen()
    console.print(Panel.fit(
        "        MILESTONE PROJECT - LOGIN DATA CLEANING        ",
        style="bold white on blue",
        border_style="blue"
    ))
    console.print()

    for row in reader:
        total_rows += 1
        issues = set()

        user_value = (row.get("user_id") or "").strip().lower()
        row["user_id"] = user_value  # write normalized value back so clean output reflects it
        ip_value = (row.get("ip_address") or "").strip()
        row["ip_address"] = ip_value  # write normalized value back so clean output reflects it
        ts_value = (row.get("timestamp") or "").strip()
        status_value = (row.get("status") or "").strip().upper()
        row["status"] = status_value  # write normalized value back so clean output reflects it

        if not user_value or not ip_value or not ts_value or not status_value:
            issues.add("missing_fields")

        if ip_value and not is_valid_ip(ip_value):
            issues.add("invalid_ip")

        parsed_ts = None
        if ts_value:
            parsed_ts = parse_timestamp(ts_value)
            if parsed_ts is None:
                issues.add("invalid_timestamp")
            else:
                row["timestamp"] = parsed_ts.strftime("%Y-%m-%d %H:%M:%S")  # standardize output format

        if status_value and not is_valid_status(status_value):
            issues.add("invalid_status")

        if user_value and ip_value and parsed_ts:
            rounded_ts = parsed_ts.replace(second=0, microsecond=0)
            dup_key = (user_value, ip_value, rounded_ts)

            if dup_key in seen:
                issues.add("duplicate")
            else:
                seen.add(dup_key)

        if issues:
            flagged_rows += 1
            for issue in issues:
                issue_counts[issue] += 1
        else:
            clean_rows.append(row)

    console.print(f"TOTAL ROWS READ: {total_rows}")
    console.print(f"{len(clean_rows)} clean vs {flagged_rows} flagged\n")


with open(OUTPUT_FILE, "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for row in clean_rows:
        writer.writerow(row)

console.print(f"Clean data written to {OUTPUT_FILE}")

console.print("DATA QUALITY REPORT")
console.print("-" * 50)

report_table = Table(box=None)
report_table.add_column("Issue Type", style="bold yellow")
report_table.add_column("Rows Affected", justify="right", style="bold red")

for issue_label, count in issue_counts.items():
    report_table.add_row(issue_label.replace("_", " ").title(), str(count))

console.print(report_table)

flagged_pct = (flagged_rows / total_rows * 100) if total_rows else 0
clean_pct = (len(clean_rows) / total_rows * 100) if total_rows else 0

console.print(Panel(
    f"Total Rows Processed: {total_rows}\n"
    f"Clean Rows Written:   {len(clean_rows)} ({clean_pct:.1f}%)\n"
    f"Flagged Rows:         {flagged_rows} ({flagged_pct:.1f}%)\n"
    f"(a flagged row is counted once here, even if it tripped multiple issue types above)",
    title="Summary",
    border_style="cyan"
))

console.save_text(REPORT_FILE)


## Findings

Running the final script against the real `messy_logins.csv` (8 rows) produced 5 clean rows and 3 flagged rows: 2 for missing fields (one blank IP, one blank timestamp) and 1 for an unrecognized status value (`N/A`). No rows were flagged for invalid IP, invalid timestamp, or duplicate, meaning every timestamp in the file parsed successfully under one of the three accepted formats, and the same user/IP pair that appeared twice in the file (`user_01` / `192.168.1.10`) was 6 minutes apart, outside the 1-minute duplicate window, so it correctly stayed clean rather than being treated as a repeat login.

The more useful result is what the write-back revisions changed about the output itself: before Revision 2 through 4, the clean CSV technically passed validation but still carried inconsistent casing, stray whitespace, and three different timestamp formats in the same column, exactly the kind of "clean" file that looks fine until something downstream tries to sort, group, or compare it. The final version guarantees that anything written to `clean_logins.csv` is not just individually valid but consistently formatted across every row.
